In [5]:
import json
import torch.nn as nn
import torch
from torch import Tensor
# from huggingface_hub import hf_hub_download
from stable_audio_tools.models.factory import create_model_from_config
# from stable_audio_tools.models.autoencoders import AudioAutoencoder


class LevoVAE(nn.Module):
    def __init__(self, device="cuda"):
        super().__init__()
        config_path = '/data/250010171/ckpts/pretrained/levo/vae/stable_audio_1920_vae.json'
        model_path = '/data/250010171/ckpts/pretrained/levo/vae/autoencoder_music_1320k.ckpt'
        with open(config_path, "r") as f:
            model_config = json.load(f)
        
        self.vae = create_model_from_config(model_config)
        #froze
        for name, param in self.vae.named_parameters():
            param.requires_grad = False
        sd = torch.load(model_path, map_location=device, weights_only=False)["state_dict"]
        self.vae.load_state_dict(sd)
        self.vae.to(device)
        self.vae.eval()

        self.sr = model_config["sample_rate"]
        self.fps = 25
    @torch.no_grad()
    def encode(self, audio: Tensor) -> Tensor:
        r"""

        Args:
            audio: (b, 2, l)
        """

        # audio_list = [a for a in audio]
        # audios = self.vae.preprocess_audio_list_for_encoder(audio_list, [self.sr] * len(audio_list))
        # latent = self.vae.encode_audio(audios)
        latent = self.vae.encode_audio(audio)
        return latent
    @torch.no_grad()
    def decode(self, latent):
        audio = self.vae.decode_audio(latent)
        return audio

    def __call__(self, audio: Tensor) -> Tensor:
        return self.encode(audio)


In [6]:
import time
import librosa
audio_path = '/data/250010171/code/AnyTrainer-midi2audio/data/debug/audio_379309.wav'
audio,_ = librosa.load(audio_path, sr=48000, mono=False, offset = 0, duration = 30)
audio = torch.from_numpy(audio).float().unsqueeze(0).unsqueeze(0)
b = 3500//(audio.shape[-1]//1920)
print(b)
audio = audio.repeat(b, 2, 1).to("cuda")
print(audio.shape)
vae = LevoVAE(device="cuda")
start_time = time.time()
latent = vae.encode(audio)
end_time = time.time()
print("vae encode time: ", end_time - start_time)    

11
torch.Size([11, 2, 595200])
vae encode time:  0.007651090621948242
